### Drivers Dimension: Silver to Gold
Join `formula1_incr.silver.drivers` with `formula1_incr.gold.ref_nationalaty_regions` into `formula1_incr.gold.dim_drivers`.

In [0]:
%run ../00-common/01.environment-config 

In [0]:
dbutils.widgets.text('p_batch_id','')
v_batch_id= dbutils.widgets.get('p_batch_id')

In [0]:
%run ../00-common/04.gold_helpers 

#### Setup
- `01.environment-config` → loads catalog name, silver/gold schema names
- `04.gold_helpers` → loads the `write_to_gold()` function

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import functions as f

#### Target Table

In [0]:

target_table = f'{catalog_name}.{gold_schema}.dim_drivers'

#### Read Source Tables
- Read `drivers` from silver (filtered by batch)
- Read `ref_nationalaty_regions` reference table from gold

In [0]:
drivers_df = (
    spark.table(f'{catalog_name}.{silver_schema}.drivers')
.filter(col('batch_id') == v_batch_id))

ref_nationality_region_df = (
    spark.read.table(f'{catalog_name}.{gold_schema}.ref_nationalaty_regions')

)

#### Join
- Left outer join drivers + regions on `nationality` to add `nationality_region` column

In [0]:
dim_drivers_df =(
    drivers_df
     .join(
        ref_nationality_region_df,
        drivers_df.nationality == ref_nationality_region_df.nationality,
        'left_outer')
    .select(
        drivers_df.driver_id,
        drivers_df.driver_name,
        drivers_df.date_of_birth,
        drivers_df.nationality,
        ref_nationality_region_df.region.alias('nationality_region'))
)

#### Write to Gold
- If table doesn't exist, creates it fresh
- If it exists, merges using `write_to_gold()`

In [0]:
write_to_gold(
    input_df= dim_drivers_df,
    target_table=target_table,
    merge_condition='t.driver_id = s.driver_id',
    columns_to_update=[
        'driver_name',
        'date_of_birth',
        'nationality',
        'nationality_region'
    ]
)

In [0]:
display(spark.table(target_table))

### Entity Relationship Diagram

The diagram below shows how the **source tables** relate to each other and how they combine into the **target dimension table** (gold layer).

```
┌─────────────────────────────────────┐         ┌─────────────────────────────────────────────┐
│       SILVER LAYER (Source)         │         │           GOLD LAYER (Target)               │
├─────────────────────────────────────┤         ├─────────────────────────────────────────────┤
│                                     │         │                                             │
│  ┌───────────────────────────┐      │         │  ┌───────────────────────────────────────┐  │
│  │  silver.constructors      │      │         │  │       gold.dim_constructors            │  │
│  ├───────────────────────────┤      │         │  ├───────────────────────────────────────┤  │
│  │ PK constructor_id         │      │         │  │  constructor_id   (from constructors)  │  │
│  │    constructor_name       │      │  JOIN   │  │  constructor_name (from constructors)  │  │
│  │    nationality ───────────│──┐   │ ──────► │  │  nationality      (from constructors)  │  │
│  │    ingestion_timestamp    │  │   │         │  │  region           (from ref table)     │  │
│  │    source_file            │  │   │         │  └───────────────────────────────────────┘  │
│  └───────────────────────────┘  │   │         │                                             │
│                                  │   │         │                                             │
└──────────────────────────────────┼───┘         └─────────────────────────────────────────────┘
                                   │
         ┌─────────────────────────┼─────┐
         │  GOLD LAYER (Reference) │     │
         ├─────────────────────────┼─────┤
         │                         │     │
         │  ┌──────────────────────┴──┐  │
         │  │ gold.ref_nationalaty_   │  │
         │  │ regions                 │  │
         │  ├─────────────────────────┤  │
         │  │    nationality ─────────│──┘
         │  │    region               │
         │  └─────────────────────────┘
         │                               │
         └───────────────────────────────┘
```

**Relationship:** `constructors.nationality` (FK) → `ref_nationalaty_regions.nationality` (PK) — **LEFT OUTER JOIN**

**Result:** One row per constructor, enriched with geographic region. Constructors without a matching nationality retain `null` for region.